<a href="https://colab.research.google.com/github/RoseWang-web/Data_Portfolio/blob/main/Case01_high_and_low_Econ_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import polars as pl
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

# ==========================================
# 1. LOAD AND PREPROCESS DATA
# ==========================================
data_url = "https://raw.githubusercontent.com/byui-cse/cse450-course/master/data/bank.csv"

print("🔄 Loading banking dataset...")
df = pl.read_csv(data_url, infer_schema_length=None)

# Task 4 Focus: Use median to split economic environments
median_confidence = df["cons.conf.idx"].median()

high_confidence_df = df.filter(pl.col("cons.conf.idx") >= median_confidence)
low_confidence_df = df.filter(pl.col("cons.conf.idx") < median_confidence)

def prepare_model_data(data_frame: pl.DataFrame, target_column: str = "y"):
    """Maps target label to binary integers and applies one-hot encoding natively in Polars."""
    target = data_frame.select(
        pl.col(target_column).map_elements(lambda val: 1 if val == "yes" else 0, return_dtype=pl.Int8)
    )
    features_df = data_frame.drop(target_column)
    categorical_cols = [col for col in features_df.columns if features_df[col].dtype == pl.String]
    features_encoded = features_df.to_dummies(columns=categorical_cols)

    return features_encoded, target

X_high, y_high = prepare_model_data(high_confidence_df)
X_low, y_low = prepare_model_data(low_confidence_df)

# ==========================================
# 2. CREATE BALANCED TRAIN/TEST SPLITS
# ==========================================
X_train_high, X_test_high, y_train_high, y_test_high = train_test_split(
    X_high.to_pandas(),
    y_high.to_pandas().to_numpy().ravel(),
    test_size=0.20,
    random_state=42,
    stratify=y_high.to_pandas()
)

X_train_low, X_test_low, y_train_low, y_test_low = train_test_split(
    X_low.to_pandas(),
    y_low.to_pandas().to_numpy().ravel(),
    test_size=0.20,
    random_state=42,
    stratify=y_low.to_pandas()
)

# ==========================================
# 3. ROBUST PIPELINE EVALUATION FUNCTION
# ==========================================
def evaluate_and_summarize(model, X_train, X_test, y_train, y_test, model_name: str, baseline_counts=None):
    """Trains model, calculates metrics, and extracts top driving data attributes."""
    # Fit the machine learning model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Calculate operational metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)

    # Format and display executive report block
    print("\n" + "="*50)
    print(f" 📑 EXECUTIVE SUMMARY FOR: {model_name.upper()}")
    print("="*50)

    if baseline_counts is not None:
        print(f"📊 Dataset Baseline Counts:\n{baseline_counts}\n---")

    print(f"Performance Scores:")
    print(f"  • Accuracy:  {accuracy:.2f}")
    print(f"  • Precision: {precision:.2f}")
    print(f"  • Recall:    {recall:.2f}")
    print("---")

    # Feature Importance analysis extraction
    if hasattr(model, 'feature_importances_'):
        importances = model.feature_importances_
        feature_names = X_train.columns

        # Sort features based on how much they impact prediction accuracy
        importance_df = pd.DataFrame({'Attribute': feature_names, 'Impact_Score': importances})
        top_5 = importance_df.sort_values(by='Impact_Score', ascending=False).head(5)

        print("💡 Top 5 Most Impactful Attributes Driving Customer Subscriptions:")
        for idx, row in top_5.iterrows():
            print(f"  [{row['Impact_Score']:.4f}] ➔ {row['Attribute']}")
    else:
        print("❌ Feature importances unavailable for this classifier profile.")
    print("="*50)

# ==========================================
# 4. RUN EXECUTIVES MODELS AND SUMMARIES
# ==========================================
# High Confidence Metrics & Importance
high_baseline = y_high.group_by("y").count()
rf_high = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
evaluate_and_summarize(rf_high, X_train_high, X_test_high, y_train_high, y_test_high, "Random Forest (High Econ Scenario)", high_baseline)

dt_high = DecisionTreeClassifier(max_depth=5, random_state=42)
evaluate_and_summarize(dt_high, X_train_high, X_test_high, y_train_high, y_test_high, "Decision Tree (High Econ Scenario)")

# Low Confidence Metrics & Importance
low_baseline = y_low.group_by("y").count()
rf_low = RandomForestClassifier(max_depth=5, random_state=42, n_estimators=100)
evaluate_and_summarize(rf_low, X_train_low, X_test_low, y_train_low, y_test_low, "Random Forest (Low Econ Scenario)", low_baseline)

dt_low = DecisionTreeClassifier(max_depth=5, random_state=42)
evaluate_and_summarize(dt_low, X_train_low, X_test_low, y_train_low, y_test_low, "Decision Tree (Low Econ Scenario)")

🔄 Loading banking dataset...


/tmp/ipykernel_24714/410228936.py:103: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  high_baseline = y_high.group_by("y").count()



 📑 EXECUTIVE SUMMARY FOR: RANDOM FOREST (HIGH ECON SCENARIO)
📊 Dataset Baseline Counts:
shape: (2, 2)
┌─────┬───────┐
│ y   ┆ count │
│ --- ┆ ---   │
│ i8  ┆ u32   │
╞═════╪═══════╡
│ 0   ┆ 17337 │
│ 1   ┆ 2529  │
└─────┴───────┘
---
Performance Scores:
  • Accuracy:  0.90
  • Precision: 0.71
  • Recall:    0.31
---
💡 Top 5 Most Impactful Attributes Driving Customer Subscriptions:
  [0.2205] ➔ emp.var.rate
  [0.1432] ➔ nr.employed
  [0.1217] ➔ euribor3m
  [0.0792] ➔ pdays
  [0.0714] ➔ poutcome_success

 📑 EXECUTIVE SUMMARY FOR: DECISION TREE (HIGH ECON SCENARIO)
Performance Scores:
  • Accuracy:  0.89
  • Precision: 0.59
  • Recall:    0.41
---
💡 Top 5 Most Impactful Attributes Driving Customer Subscriptions:
  [0.7912] ➔ emp.var.rate
  [0.1244] ➔ pdays
  [0.0182] ➔ euribor3m
  [0.0140] ➔ day_of_week_mon
  [0.0104] ➔ nr.employed


/tmp/ipykernel_24714/410228936.py:111: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  low_baseline = y_low.group_by("y").count()



 📑 EXECUTIVE SUMMARY FOR: RANDOM FOREST (LOW ECON SCENARIO)
📊 Dataset Baseline Counts:
shape: (2, 2)
┌─────┬───────┐
│ y   ┆ count │
│ --- ┆ ---   │
│ i8  ┆ u32   │
╞═════╪═══════╡
│ 0   ┆ 15524 │
│ 1   ┆ 1679  │
└─────┴───────┘
---
Performance Scores:
  • Accuracy:  0.91
  • Precision: 0.69
  • Recall:    0.05
---
💡 Top 5 Most Impactful Attributes Driving Customer Subscriptions:
  [0.1998] ➔ cons.conf.idx
  [0.1179] ➔ euribor3m
  [0.0863] ➔ pdays
  [0.0822] ➔ cons.price.idx
  [0.0656] ➔ poutcome_success

 📑 EXECUTIVE SUMMARY FOR: DECISION TREE (LOW ECON SCENARIO)
Performance Scores:
  • Accuracy:  0.91
  • Precision: 0.69
  • Recall:    0.10
---
💡 Top 5 Most Impactful Attributes Driving Customer Subscriptions:
  [0.4968] ➔ cons.conf.idx
  [0.1551] ➔ euribor3m
  [0.0805] ➔ pdays
  [0.0691] ➔ age
  [0.0435] ➔ job_retired
